# 11 — Consumption / Production Operations

Verify that the new `OperationType.CONSUMPTION` and `OperationType.PRODUCTION` values let role derivation produce `SINK` and `SOURCE` symmetrically with the existing `STORAGE` and `TRANSSHIPMENT` rules.

Motivating use case: a non-bike-sharing domain (e.g. gas delivery) where the customer is a `Facility` node that consumes commodity and removes it from the network. Without `consumption`, a customer could not declare its `SINK` role through the operation set — only via the L3 bike-sharing `DEFAULT_ROLES` mapping, which does not contain a `customer` entry.

Sections:
1. Direct `derive_roles` smoke checks for the four new scenarios.
2. `derive_facility_roles` on a tiny gas-style topology (depot + customer).
3. Symmetric check with `production` on a producer node.
4. Regression: bike-sharing `DEFAULT_ROLES` unchanged when the new operations are absent.

In [ ]:
from gbp.core.enums import FacilityRole, FacilityType, OperationType
from gbp.core.roles import DEFAULT_ROLES, derive_roles

roles_consumer = derive_roles("customer", {OperationType.CONSUMPTION.value})
roles_producer = derive_roles("oil_well", {OperationType.PRODUCTION.value})
roles_hybrid = derive_roles(
    "hybrid_node",
    {OperationType.CONSUMPTION.value, OperationType.PRODUCTION.value},
)

assert FacilityRole.SINK in roles_consumer and FacilityRole.SOURCE not in roles_consumer
assert FacilityRole.SOURCE in roles_producer and FacilityRole.SINK not in roles_producer
assert {FacilityRole.SINK, FacilityRole.SOURCE} <= roles_hybrid

{"consumer": roles_consumer, "producer": roles_producer, "hybrid": roles_hybrid}

## Gas-style topology: depot + customer

Two facilities of L2 types (`depot`, `customer`) that are not in `DEFAULT_ROLES`. The depot supports `receiving + storage + dispatch`; the customer supports `receiving + consumption`. After `derive_facility_roles`:

- depot → `STORAGE` (from `storage` op) and `TRANSSHIPMENT` (from `receiving + dispatch`),
- customer → `SINK` (from `consumption`).

No bike-sharing default applies, so every role in the result is operation-derived.

In [ ]:
import pandas as pd

from gbp.build.defaults import derive_facility_roles

facilities_gas = pd.DataFrame({
    "facility_id": ["depot1", "customer1"],
    "facility_type": ["depot_l2", "customer"],
})
operations_gas = pd.DataFrame({
    "facility_id": [
        "depot1", "depot1", "depot1",
        "customer1", "customer1",
    ],
    "operation_type": [
        "receiving", "storage", "dispatch",
        "receiving", "consumption",
    ],
    "enabled": [True] * 5,
})

roles_gas = derive_facility_roles(facilities_gas, operations_gas)
customer_roles = set(
    roles_gas.loc[roles_gas["facility_id"] == "customer1", "role"]
)
depot_roles = set(
    roles_gas.loc[roles_gas["facility_id"] == "depot1", "role"]
)

assert customer_roles == {FacilityRole.SINK.value}
assert depot_roles == {FacilityRole.TRANSSHIPMENT.value}

roles_gas

## Symmetric check: producer node

Mirror of the previous block with a producer (e.g. oil well, factory). The node has `production + dispatch` and is expected to derive `SOURCE` from the `production` op and `TRANSSHIPMENT` is not added because there is no `receiving`.

In [ ]:
facilities_producer = pd.DataFrame({
    "facility_id": ["well1"],
    "facility_type": ["oil_well"],
})
operations_producer = pd.DataFrame({
    "facility_id": ["well1", "well1"],
    "operation_type": ["production", "dispatch"],
    "enabled": [True, True],
})

roles_producer_table = derive_facility_roles(facilities_producer, operations_producer)
well_roles = set(
    roles_producer_table.loc[roles_producer_table["facility_id"] == "well1", "role"]
)

assert well_roles == {FacilityRole.SOURCE.value}

roles_producer_table

## Regression: bike-sharing defaults are untouched

Stations without `consumption` / `production` operations must keep the historical role set `{SOURCE, SINK, STORAGE}` provided by `DEFAULT_ROLES`. This guards against accidental coupling between the new operations and existing bike-sharing fixtures.

In [ ]:
facilities_bike = pd.DataFrame({
    "facility_id": ["station1"],
    "facility_type": [FacilityType.STATION.value],
})
operations_bike = pd.DataFrame({
    "facility_id": ["station1", "station1", "station1"],
    "operation_type": ["receiving", "storage", "dispatch"],
    "enabled": [True, True, True],
})

roles_bike = derive_facility_roles(facilities_bike, operations_bike)
station_roles = set(
    roles_bike.loc[roles_bike["facility_id"] == "station1", "role"]
)
expected_station_roles = {role.value for role in DEFAULT_ROLES[FacilityType.STATION.value]} | {
    FacilityRole.TRANSSHIPMENT.value
}

assert station_roles == expected_station_roles

{"station1": station_roles, "expected": expected_station_roles}